# Image Augmentation

Our baseline network overfit heavily - it memorised the exact training images. Here we add Image Augmentation, randomly changing each training image slightly every epoch so the network never sees the exact same image twice. As with Dropout, everything else stays the same as the baseline (no Dropout is applied here), so we can measure the effect of Image Augmentation on its own.

In [ ]:
# reload src/confi.py automatically whenever it changes
%load_ext autoreload
%autoreload 2

# import project settings
import sys
sys.path.append("..")
from src import config
from src.plots import plot_training_history

# import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay

# make results repeatable
tf.keras.utils.set_random_seed(config.RANDOM_SEED)

## Data Pipeline

Load images from the training, validation & test folders - same as the baseline.

In [ ]:
# training data
training_set = tf.keras.utils.image_dataset_from_directory(
    config.TRAINING_DIR,
    batch_size = config.BATCH_SIZE,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    label_mode = 'categorical',
    shuffle = True,
    seed = config.RANDOM_SEED
)
# validation data
validation_set = tf.keras.utils.image_dataset_from_directory(
    config.VALIDATION_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = False
)
# test data
test_set = tf.keras.utils.image_dataset_from_directory(
    config.TEST_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = False
)

# class names
class_names = training_set.class_names
print(class_names)


## Image Augmentation Layers

Each training image is randomly transformed every epoch. Each setting is chosen to create images that could realistically appear in a product catalogue:

* **Horizontal flip** - a mirrored product is still the same product (no vertical flip - upside-down products never appear in a catalogue)
* **Rotation** - up to ±18 degrees, as products are photographed fairly straight
* **Zoom** - up to 10%, as products are not always the same size in the frame
* **Shift** - up to 10% up/down & left/right, as products are not always perfectly centred

Gaps created by rotating or shifting are filled with the nearest pixel - usually the white background.


In [ ]:
# image augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.Input(shape=(config.IMG_HEIGHT, config.IMG_WIDTH, config.NUM_CHANNELS)),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05, fill_mode='nearest'),
    tf.keras.layers.RandomZoom(0.1, fill_mode='nearest'),
    tf.keras.layers.RandomTranslation(0.1,0.1, fill_mode='nearest')
], name = 'data_augmentation')

### Augmentation Preview

One training image, shown with 8 random augmented versions - each still looks like a realistic product photo.

In [ ]:
# take one training image and show 8 random augmented versions of it
for images, labels in training_set.take(1):
    original_image = images[1]

fig, axes = plt.subplots(2, 4, figsize=(8, 6))

for ax in axes.flat:
    image_batch = tf.expand_dims(original_image, axis=0) 
    augmented_image = data_augmentation(image_batch, training=True)[0]  # training=True switches augmentation on
    ax.imshow(augmented_image.numpy().astype("uint8"))
    ax.axis("off")

plt.suptitle("One training image - 8 random augmented versions")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "augmentation-examples.png", dpi=150)
plt.show()

## Network Architecture

The same architecture as the baseline, with the augmentation layers added at the start of the network. No Dropout is applied, so any change in results comes from Image Augmentation alone.

In [ ]:
# network architecture (Functional API) - baseline + Image Augmentation

# input: image of 80 x 60 pixels, 3 color channels
inputs = tf.keras.Input(shape=(config.IMG_HEIGHT,config.IMG_WIDTH, config.NUM_CHANNELS))

# image augmentation
x = data_augmentation(inputs)

# rescale pixels to 0-1
x = tf.keras.layers.Rescaling(1./255)(x)

# convolutional block 1
x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)

# convolutional block 2
x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)

# flatten the feature maps into one long list of numbers
x = tf.keras.layers.Flatten()(x)

# dense layers
x = tf.keras.layers.Dense(32,activation='relu')(x)

# output layer
outputs = tf.keras.layers.Dense(config.NUM_CLASSES, activation='softmax')(x)

# connect inputs to outputs to create the model
model = tf.keras.Model(inputs=inputs, outputs=outputs, name='augmentation_cnn')

# view network architecture
model.summary()

In [ ]:
# compile the model
model.compile(loss = 'categorical_crossentropy',
             optimizer = 'adam',
             metrics = ['accuracy'])


## Training The Network

Train for 50 epochs, saving the best model (highest validation accuracy) to a new file so earlier models are kept for comparison.

In [ ]:
# save the best model
model_filename = config.MODELS_DIR/"fashion_cnn_v03_augmentation.keras"

save_best_model = tf.keras.callbacks.ModelCheckpoint(
    filepath=model_filename,
    monitor= 'val_accuracy',
    mode= 'max',
    save_best_only= True,
    verbose=0
)

# train the network
history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=config.NUM_EPOCHS,
    callbacks= [save_best_model],
    shuffle= False
)

## Analysis Of Training Results

Compare training & validation performance epoch by epoch - has Image Augmentation reduced the gap seen in the baseline?

In [ ]:
# plot training results
plot_training_history(history,
                      title="Augmentation Network",
                      save_path=config.FIGURES_DIR / "augmentation-accuracy-plot.png")

## Performance On The Test Set

Measure how the best saved model performs on images it has never seen, overall and for each class.

In [ ]:
# load the best model saved during training (highest validation accuracy)
best_model = tf.keras.models.load_model(model_filename)

# evaluate on the test set
test_loss, test_accuracy = best_model.evaluate(test_set, verbose=0)

print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test loss: {test_loss:.4f}")

## Error Analysis

Look at the network's mistakes and how confident it was - and test whether that confidence could be used to decide which products to auto-tag.

In [ ]:
# predicted probabilities for every test image, one column per class
probabilities_df = pd.DataFrame(best_model.predict(test_set, verbose=0), columns=class_names)

# true labels for every test image (same order, because test_set has shuffle=False)
actual_df = pd.DataFrame(np.concatenate([labels for _, labels in test_set]), columns=class_names)

# create dataframe to analyse
predictions_df = pd.DataFrame({"actual_label": actual_df.idxmax(axis=1),
                               "predicted_label": probabilities_df.idxmax(axis=1),
                               "predicted_probability": probabilities_df.max(axis=1),
                               "filename": [Path(path).name for path in test_set.file_paths]})

predictions_df["correct"] = np.where(predictions_df["actual_label"] == predictions_df["predicted_label"], 1, 0)


print(predictions_df["correct"].mean())

In [ ]:
# confusion matrix heatmap - rows = actual class, columns = predicted class
fig, ax = plt.subplots(figsize=(8, 7))

ConfusionMatrixDisplay.from_predictions(
    predictions_df["actual_label"],
    predictions_df["predicted_label"],
    labels=config.CLASSES,       # look-alike pairs next to each other
    cmap="Blues",
    colorbar=False,
    ax=ax
)

ax.set_title("Augmentation Network - Test Set Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "augmentation-confusion-matrix.png", dpi=150)
plt.show()

# accuracy for each class
class_accuracy = predictions_df.groupby("actual_label")["correct"].mean().sort_values()
print(class_accuracy.round(2))

In [ ]:
# the model's mistakes, most confident first
mistakes = predictions_df.loc[predictions_df["correct"] == 0].sort_values("predicted_probability", ascending=False)
print(f"Number of mistakes: {len(mistakes)}")

# typical confidence when right (1) vs wrong (0)
print(predictions_df.groupby("correct")["predicted_probability"].median())

mistakes.head(10)

In [ ]:
# what if we only auto-tag products the model is at least 90% confident?
threshold = 0.90

confident = predictions_df.loc[predictions_df["predicted_probability"] >= threshold]
not_confident = predictions_df.loc[predictions_df["predicted_probability"] < threshold]

print(f"Auto-tagged: {len(confident)} of {len(predictions_df)} products ({len(confident) / len(predictions_df):.0%})")
print(f"Accuracy on auto-tagged products: {confident['correct'].mean():.1%}")
print(f"Sent to a person to check: {len(not_confident)} products")

## Summary

**Change from Baseline**

* Added Image Augmentation layers at the start of the network (horizontal flip, rotation up to ±18°, zoom & shift up to 10%) - no Dropout, everything else unchanged
* Parameters unchanged at 317,574 (augmentation layers have nothing to learn)

**Training**

* Best validation accuracy: **89.3%** at epoch 26
* No clear overfitting - training & validation accuracy stay close together throughout
* However, validation accuracy is unstable, swinging between 84% & 89% from epoch to epoch, with no real improvement after around epoch 20

**Test Set Performance**

| | Baseline | Dropout | Augmentation |
|---|---|---|---|
| Test accuracy | 92.3% | 91.5% | **87.7%** |
| Test loss | 0.58 | 0.38 | **0.35** |

| Class | Baseline | Dropout | Augmentation |
|---|---|---|---|
| Handbags | 99% | 99% | 99% |
| Watches | 98% | 97% | 98% |
| Shirts | 95% | 95% | 93% |
| Tshirts | 94% | 94% | 90% |
| Sports Shoes | 79% | 79% | 86% |
| Casual Shoes | 89% | 85% | 60% |

* Accuracy is clearly lower than the baseline - 526 vs 554 correct
* Most of the drop comes from Casual Shoes, with 39 of 100 predicted as Sports Shoes
* Results are unstable between runs - an earlier run of this same network scored 90.5% (543 correct), with Casual Shoes at 79%. This suggests the network lacks the capacity to reliably learn from augmented images

**Error Analysis**

* Median confidence is 99.6% on correct predictions vs 70% on mistakes - confidence clearly separates right from wrong

**Business Application**

* Auto-tagging only products with at least 90% confidence covers 66% of products at 97.5% accuracy, with the remaining 34% (207 products) sent for manual review

**Next:** `05_hyperparameter_tuning.ipynb` - use keras-tuner to find a larger architecture, combining Dropout & Image Augmentation